# Análise exploratória — Problemas no domicílio nas 12 maiores capitais (POF 2002-2003)
Notebook de exploração e validação. A lógica reutilizável está em `src/etl.py` e `src/analise.py`.
Execute a partir da **raiz do repositório** (a célula abaixo ajusta isso se você abrir dentro de `notebooks/`).

In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
import pandas as pd
from src import etl, analise
pd.set_option("display.width", 160)

## 1. Dado bruto (como vem da planilha)

In [ ]:
raw = etl.read_raw(etl.RAW_DEFAULT)
raw.iloc[:14, :6]

## 2. ETL: formato tidy + validação

In [ ]:
df = etl.run()
df.head(10)

In [ ]:
df.info()
df.isna().sum()

## 3. RQ1 — Prevalência e dispersão por problema

In [ ]:
w = analise.matriz(df)
desc = w.describe().T
desc["cv"] = desc["std"] / desc["mean"]
desc.sort_values("mean", ascending=False).round(2)

## 4. RQ2 — Índice médio por capital e por região

In [ ]:
indice = w.mean(axis=1).sort_values(ascending=False)
indice.round(2)

In [ ]:
indice.groupby(indice.index.map(etl.REGIAO)).mean().sort_values(ascending=False).round(2)

In [ ]:
z = (w - w.mean()) / w.std()
z[z.abs() >= 1.8].stack().round(2)   # valores extremos

## 5. RQ3 — Correlação entre problemas

In [ ]:
w.corr(method="spearman").round(2)

In [ ]:
from scipy import stats
stats.spearmanr(w["goteira"], w["madeira_deteriorada"])

## 6. Figuras do artigo

In [ ]:
analise.main()   # grava PNGs e estatisticas.json em docs/assets/

## 7. Observações
- Dados agregados (n = 12 capitais): correlações são **ecológicas** e instáveis.
- Informante único por família; respostas binárias e autodeclaradas.